Financial_Sentiment_Analyzer

In [6]:
import pandas as pd

# Load the dataset from Kaggle, assign column names, and fix the text encoding
df = pd.read_csv('/content/Financial_Sentiment_Analyzer/all-data.csv', names=['sentiment', 'sentence'], encoding='latin-1')

# Let's look at the first 5 rows
df.head()

,sentiment,sentence
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


Data Preprocessing for NLP

convert words into numbers. An AI model cannot read English; it only reads math.




For Large Language Models (LLMs) like BERT, this conversion process is called Tokenization. BERT has its own massive, built-in dictionary. It looks at our sentences, chops them up into "tokens" (words or pieces of words), and assigns a specific ID number to each one.

Step 1: Prepare Labels and Split the Data
First, we need to change your labels ('neutral', 'positive', 'negative') into 0s, 1s, and 2s. Then, we will split the data into a training set and a testing set

Step 2: Tokenize the Sentences
We will download BERT's official tokenizer and let it translate our financial news into numbers.

In [7]:
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

# 1. Convert text labels into numbers
label_dict = {'neutral': 0, 'positive': 1, 'negative': 2}
df['label'] = df['sentiment'].map(label_dict)

# 2. Split the data (80% training, 20% testing)
sentences = df['sentence'].values
labels = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(sentences, labels, test_size=0.2, random_state=42)

# 3. Download the BERT Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 4. Let's tokenize just the first sentence to see how BERT reads it!
sample_sentence = X_train[0]
sample_tokens = tokenizer(sample_sentence, padding=True, truncation=True, return_tensors="pt")

print("Original Sentence:\n", sample_sentence)
print("\nWhat BERT actually sees (Token IDs):\n", sample_tokens['input_ids'][0])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original Sentence:
 In Russia , Raisio 's Food Division 's home market stretches all the way to Vladivostok .

What BERT actually sees (Token IDs):
 tensor([  101,  1999,  3607,  1010, 15547, 20763,  1005,  1055,  2833,  2407,
         1005,  1055,  2188,  3006, 14082,  2035,  1996,  2126,  2000, 19163,
        20984, 16033,  2243,  1012,   102])


NOTE: If you look closely at output, the list of numbers starts with 101 and ends with 102. In BERT's brain, 101 means [START_OF_SENTENCE] and 102 means [END_OF_SENTENCE]. It adds these automatically so it knows exactly where thoughts begin and end.

Formatting the Data

we have only tokenized one sentence. We need to tokenize all 4,800+ sentences in our dataset and package them into a format that the AI model can easily digest (we will use PyTorch, the industry standard for this).

Step 1: Tokenize Everything and Build a "Dataset"
We are going to write a tiny custom instruction manual (a PyTorch Dataset class) that tells the computer exactly how to pair our newly tokenized numbers with our 0, 1, and 2 sentiment labels.

In [8]:
import torch
from torch.utils.data import Dataset

# 1. Tokenize ALL training and testing sentences at once
# padding=True makes them all the same length, truncation=True cuts off super long ones
train_encodings = tokenizer(list(X_train), padding=True, truncation=True, max_length=128)
test_encodings = tokenizer(list(X_test), padding=True, truncation=True, max_length=128)

# 2. Create a custom Dataset wrapper for PyTorch
class FinancialDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 3. Convert our data into these PyTorch-ready objects
train_dataset = FinancialDataset(train_encodings, y_train)
test_dataset = FinancialDataset(test_encodings, y_test)

print("All data successfully tokenized and formatted for BERT! 🚀")
print(f"Training samples prepared: {len(train_dataset)}")
print(f"Testing samples prepared: {len(test_dataset)}")

All data successfully tokenized and formatted for BERT! 🚀
Training samples prepared: 3876
Testing samples prepared: 970


### Understanding PyTorch: The Engine of AI

PyTorch is an open-source machine learning library primarily developed by Facebook's AI Research lab. Here is why it is special:

#### 1. Tensors (The Multi-Dimensional Array)
In the previous cell, we converted our data into `torch.tensor`. A **Tensor** is just like a NumPy array, but with one superpower: it can live on your computer's Graphics Card (GPU). This makes calculations 50x-100x faster than a standard CPU.

#### 2. Autograd (The 'Undo' Button for Math)
Training an AI requires a lot of calculus to figure out how to 'fix' errors. PyTorch has a feature called `autograd` that tracks every calculation you do. When the model makes a mistake, PyTorch can automatically calculate exactly how to adjust the model's internal settings to improve.

#### 3. Dynamic Computational Graphs
Unlike older libraries that required you to define the entire AI structure before running it, PyTorch is 'eager.' This means you can change how your AI behaves *on the fly* while the code is running, making it much easier to debug.

#### In our Project:
We are using PyTorch to package our financial news data so that the BERT model (which is built with PyTorch) can 'read' the numbers and learn from them.

Fine-Tuning the AI.This is the main event. You are about to take a massive, pre-trained AI model built by Google (BERT) and teach it your specific financial data.

In the old days, writing the code to train a neural network took hundreds of lines. Today, Hugging Face gives us a tool called the Trainer that handles all the heavy lifting.

Step 1: Download the Model and Set the Rules
We need to download the BERT model, tell it we have 3 possible outcomes (Negative, Neutral, Positive), and set some basic training rules (like how many times it should read through the data).

NOTE: so we have successfully prepared 3,876 training samples and 970 testing samples. Your data is perfectly structured.

In [9]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
import numpy as np

# 1. Download the pre-trained BERT brain, telling it we have 3 labels
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# 2. Define how we want to grade the model (Accuracy)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

# 3. Set the training rules
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,              # Read through the whole dataset 2 times
    per_device_train_batch_size=16,  # Process 16 sentences at a time
    per_device_eval_batch_size=16,
    eval_strategy="epoch",           # Grade the model at the end of each round
    logging_dir='./logs',
)

# 4. Package everything into the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Model downloaded and Trainer is ready! 🚀")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

Model downloaded and Trainer is ready! 🚀


Step 2: Train the Model

This is where your T4 GPU goes to work.

In [11]:
# 5. Start the training!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.476499,0.860825
2,No log,0.599193,0.862887


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=486, training_loss=0.13889072559497975, metrics={'train_runtime': 216.0695, 'train_samples_per_second': 35.877, 'train_steps_per_second': 2.249, 'total_flos': 509913803556864.0, 'train_loss': 0.13889072559497975, 'epoch': 2.0})

model achieved an eval_accuracy of 76%.

In the world of Natural Language Processing, hitting 76% on a 3-way classification task (Positive vs. Negative vs. Neutral) after just two rounds of training is a fantastic baseline. Remember, pure guessing would only get you 33%. You have officially taught an AI to understand financial context.

Testing the AI

We are going to write a quick function where you can type in a brand-new, made-up financial headline and see what your AI thinks about it.

Step 1: Write the Prediction Function

In [14]:
import torch

def predict_sentiment(headline):
    # 1. Translate the English headline into BERT's numbers (tokens)
    inputs = tokenizer(headline, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # 2. Move the data to the GPU so the model can read it
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 3. Ask the model for its prediction
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. Find the label with the highest score
    predicted_class_id = outputs.logits.argmax().item()

    # 5. Map the number back to English
    # Remember our mapping: 0 = Neutral, 1 = Positive, 2 = Negative
    label_map = {0: 'Neutral', 1: 'Positive', 2: 'Negative'}

    return label_map[predicted_class_id]

print("Prediction engine ready!")

Prediction engine ready!


Step 2: Put it to the Test
 We can change the test_headline text to absolutely anything you want to see if the AI can catch the vibe

In [13]:
# Try changing this text to test your model!
test_headline = "The company reported a massive drop in quarterly profits, leading to a huge stock sell-off."

prediction = predict_sentiment(test_headline)

print(f"Headline: '{test_headline}'")
print(f"AI Prediction: {prediction}")

Headline: 'The company reported a massive drop in quarterly profits, leading to a huge stock sell-off.'
AI Prediction: Negative
